# 03 — Functions and Dictionaries

**Goal of this notebook:** be able to read any function definition in `engine/`, understand what it takes in and what it returns, and read any config dict (like `DEFAULT_GATE1_WRAPPER` or `MFFU_PHASE1_100K`) and know what every key means.

**What's new vs notebooks 01–02:** until now your code lived in flat cells. Functions let you bundle a chunk of logic, give it a name, and reuse it. Dicts let you bundle related values under named keys — the way every config in this codebase is structured.

**The bridge from notebook 02:** every accumulator pattern you wrote can be wrapped in a function. The body is the same loop you already know — the wrapper just gives it a name and a return value. That's most of what `engine/metrics.py` is.

**How to use:** same as before. Predict before running. Type your own code in the empty cells. No copy-paste.

---
## 1. What is a function?

**Beginner terms:** a function is a named recipe. You give it ingredients (arguments), it does some work, and gives you back a result (the return value).

**Why we use them — three reasons:**

1. **Reuse.** If you compute profit factor in three notebooks, you don't want to rewrite the loop three times. Define `compute_pf(pnls)` once, call it three times.
2. **Naming intent.** A 6-line accumulator loop is *what*. A function called `compute_profit_factor` is *why*. The name tells the next reader (often: future-you) what the loop is for.
3. **Testability.** A function with a clear input and output can be tested in isolation. A loop buried in the middle of a 200-line script can't.

**The syntax:**

```python
def function_name(arg1, arg2):
    # body — indented, like inside a for-loop
    result = arg1 + arg2
    return result
```

Four pieces: the keyword `def`, a name, a parenthesised list of arguments, and a colon. Everything indented after the colon is the body. `return X` ends the function and sends `X` back to whoever called it.

In [ ]:
# Define a function. This cell DEFINES it — it doesn't call it yet.
def compute_pnl(entry_price, exit_price, contracts, point_value):
    points_gained = exit_price - entry_price
    pnl = points_gained * contracts * point_value
    return pnl

In [ ]:
# Now call it. Each call runs the body once with those inputs.
trade1 = compute_pnl(15000.00, 15042.50, 2, 20)
trade2 = compute_pnl(15100.00, 15067.50, 3, 20)   # short trade — what sign will this be? predict.
print("trade1:", trade1)
print("trade2:", trade2)

**What just happened:**

- Defining a function does *not* run it. It only registers the recipe.
- Calling it (`compute_pnl(15000.00, ...)`) runs the body with those specific values plugged into the argument names.
- The `return pnl` line hands the result back. `trade1 = compute_pnl(...)` captures whatever was returned.
- `trade2` is negative because we passed the prices in long-trade order to a long-trade formula, but the price went down. To handle shorts properly the function would need a `direction` argument — a good extension exercise.

**Subtle but important:** the names `entry_price`, `exit_price` etc. *only exist inside the function*. After the call ends, they're gone. This is called *scope*. It's why two different functions can both have an argument named `pnls` without clashing.

---
## 2. `print` is not `return`

This is the single most common beginner confusion with functions. They look similar inside a notebook because both display values. They are not the same thing.

- `print(x)` *displays* x to the screen. It does not give x back to the caller.
- `return x` *gives x back* to whoever called the function. It does not display anything.

If your function only `print`s and doesn't `return`, the caller has nothing to capture in a variable.

In [ ]:
def add_print(a, b):
    print(a + b)            # displays it
    # no return — the function ends here, returning None implicitly

def add_return(a, b):
    return a + b            # hands it back

x = add_print(2, 3)         # prints 5, but x captures nothing useful
y = add_return(2, 3)        # prints nothing, but y captures 5

print("x is:", x)           # what type is this? predict before running.
print("y is:", y)
print("x + y would be:", )   # try uncommenting `x + y` — it'll error. Why?

When a function ends without an explicit `return`, Python returns the special value `None`. `None` is its own type — it means "nothing here." You'll see it everywhere; recognise it. Trying to do arithmetic with `None` is a common error.

---
## 3. Default arguments

You can give an argument a default value. Callers can override it or leave it out.

```python
def compute_pnl(entry_price, exit_price, contracts, point_value=20):
    ...
```

Now `compute_pnl(15000, 15050, 2)` works — `point_value` defaults to 20. But `compute_pnl(15000, 15050, 2, point_value=5)` overrides it (useful if you ever model micro NQ, which has a $5 point value).

**Why this is everywhere in `engine/`:** the backtester takes ~15 parameters. Most have sensible defaults. You override only the ones that differ from the standard run. This is what `_metrics(equity, trades, bars_per_year=252)` style signatures are doing throughout `engine/metrics.py`.

In [ ]:
def compute_pnl(entry_price, exit_price, contracts, point_value=20):
    return (exit_price - entry_price) * contracts * point_value

# call with default point_value
print(compute_pnl(15000, 15050, 2))

# call overriding it (micro NQ — $5 per point)
print(compute_pnl(15000, 15050, 2, point_value=5))

---
## 4. Functions wrap accumulators (the bridge)

Take this loop you've written several times:

```python
total = 0
for pnl in pnls:
    total += pnl
```

Wrap it in a function:

```python
def total_pnl(pnls):
    total = 0
    for pnl in pnls:
        total += pnl
    return total
```

Same loop. The function gives it a name (`total_pnl`), a clear input (`pnls`), and a clean output (`total` via `return`). Now anywhere you have a list of pnls you can write `total_pnl(my_pnls)` and the loop runs.

**This is the single biggest pattern in `engine/metrics.py`.** Almost every metric function is exactly this shape: take some sequence in, run an accumulator over it, return the result. Read `_trade_metrics` (`engine/metrics.py:149`) once you've finished this notebook and you'll see it.

In [ ]:
# Run this. Then call total_pnl on the test list.
def total_pnl(pnls):
    total = 0
    for pnl in pnls:
        total += pnl
    return total

test_pnls = [120, -80, 250, -50, 300]
print(total_pnl(test_pnls))
print(total_pnl([100, 200, 300]))    # different list, same function — that's the point of functions

---
## 5. What is a dictionary?

**Beginner terms:** a dict is a labelled lookup. Like a list, but instead of position-numbers (`pnls[0]`, `pnls[1]`) you use named keys (`config['stop_atr']`, `config['tp_atr']`).

**The syntax:**

```python
wrapper = {
    "stop_atr": 1.5,
    "tp_atr": 2.0,
    "costs_rt": 5.0,
    "eod_exit": True,
}
```

Curly braces. Each entry is `key: value`, comma-separated. Keys are usually strings.

**Access** with square brackets: `wrapper["stop_atr"]` returns `1.5`.

**Why this is everywhere in the codebase:** every configuration in `engine/` is a dict. The default Gate 1 execution wrapper, the MFFU prop firm rules, the threshold sets — all dicts. The reason is simple: a dict is *self-documenting*. Compare:

```python
# without a dict — what does each number mean?
run(1.5, 2.0, 5.0, True)

# with a dict — the names tell you
run(stop_atr=1.5, tp_atr=2.0, costs_rt=5.0, eod_exit=True)
```

When you've got 15 parameters that need to travel together, bundling them in a dict keeps them organised and named.

In [ ]:
# A simplified version of DEFAULT_GATE1_WRAPPER from engine/gate1.py
wrapper = {
    "stop_atr": 1.5,
    "tp_atr": 2.0,
    "costs_rt": 5.0,
    "eod_exit": True,
}

print(wrapper["stop_atr"])
print(wrapper["eod_exit"])
print("len:", len(wrapper))    # number of entries

In [ ]:
# Add or change entries by assignment:
wrapper["max_holding_bars"] = 26    # adds a new key
wrapper["costs_rt"] = 6.0           # overwrites existing
print(wrapper)

---
## 6. `.get()` — the safer access

`wrapper["missing_key"]` raises `KeyError` and crashes. Sometimes you want a fallback instead.

`.get(key, default)` returns the value if the key exists, else returns the default.

```python
wrapper.get("stop_atr", 1.0)        # → 1.5  (key exists)
wrapper.get("missing_thing", 1.0)   # → 1.0  (key absent → default)
wrapper.get("missing_thing")        # → None (no default given)
```

You'll see this all over `engine/metrics.py` (search for `.get(` later — every threshold check uses it). Reason: a config dict from one caller might not have every key the function knows about; `.get()` handles missing keys gracefully.

In [ ]:
print(wrapper.get("stop_atr", 1.0))
print(wrapper.get("slippage_ticks", 0))    # not in the dict → returns the default 0
print(wrapper.get("slippage_ticks"))       # no default given → returns None

---
## 7. Iterating a dict

Three common loop forms over a dict:

```python
for key in wrapper:                  # just keys (default)
    print(key)

for value in wrapper.values():       # just values
    print(value)

for key, value in wrapper.items():   # both, as a pair — the most useful one
    print(key, "->", value)
```

`.items()` is the workhorse. Use it whenever you want to do something with each key/value pair.

In [ ]:
for key, value in wrapper.items():
    print(f"{key}: {value}")

*(That `f"{key}: {value}"` is an **f-string** — a string template that interpolates variables. Anywhere you see `f"..."`, the bits inside `{ }` get substituted with their actual values. You'll see this constantly in print/log lines.)*

---
## 8. Functions returning dicts (the deepest pattern in the codebase)

A function that does several measurements doesn't have to return one number. It can return a dict bundling all of them.

```python
def summarise(pnls):
    return {
        "n_trades": len(pnls),
        "total": sum(pnls),
        "avg": sum(pnls) / len(pnls),
    }

result = summarise([100, -50, 200])
print(result["total"])    # 250
```

This is the shape of every metric function in `engine/metrics.py`. They walk the trades, compute many things, return one dict with all of them. It keeps the function signature clean (one return value) while still delivering many results.

**Combine all three concepts** — accumulator + function wrapper + dict return — and you have the core building block of this codebase.

---
## Exercises

Type into the empty cells. Predict before running. The exercises build up: each one extends the previous concept, and the last few combine functions + dicts + accumulators.

### Exercise 1 — `compute_pf(pnls)`

Wrap the profit-factor accumulator from refresher E6 in a function called `compute_pf` that takes a list of pnls and returns the profit factor (a float).

Test it on `[100, -50, 200, -150, 80]`. Predict the answer first.

Reminder: profit factor = sum(winners) / abs(sum(losers)).

In [ ]:
# your code here


### Exercise 2 — `count_winners(pnls)` with a default threshold

Write a function `count_winners(pnls, min_pnl=0)` that returns how many trades had pnl strictly greater than `min_pnl`.

- Default `min_pnl=0` should give the standard winners count.
- Calling with `min_pnl=100` should only count trades that made more than $100.

Test on `[150, -200, 80, 250, -50, 110]`:
- `count_winners(pnls)` → predict
- `count_winners(pnls, min_pnl=100)` → predict

In [ ]:
# your code here


### Exercise 3 — build a `gate1_thresholds` dict

Construct a dict called `gate1_thresholds` with these keys (with these values):
- `min_pf` = 1.4
- `min_total_pnl` = 1500.0
- `min_trades` = 30
- `max_single_day_pct` = 0.30   *(no single day should be more than 30% of total P&L)*
- `min_half_stability` = 0.5

Then:
- print the value of `min_pf` using `[ ]` access.
- print the value of a key that doesn't exist using `.get()` with a sensible default of your choice.
- loop over the dict using `.items()` and print each as `key: value`.

In [ ]:
# your code here


### Exercise 4 — `summarise_trades(pnls)` returning a dict

Write a function `summarise_trades(pnls)` that returns a **dict** with these keys:
- `n_trades` — total number of trades
- `n_winners` — count of trades where pnl > 0
- `total_pnl` — sum of all pnls
- `gross_profit` — sum of winning pnls only
- `gross_loss` — abs(sum of losing pnls)
- `profit_factor` — gross_profit / gross_loss (be careful if gross_loss is 0 — return `0.0` in that edge case rather than crashing)

Inside, use **a single for-loop** with multiple trackers — don't call `sum()` or use list comprehensions. The point is to combine multiple accumulators in one pass.

Test on `[100, -50, 200, -150, 80, -30, 120]`. Print the returned dict.

In [ ]:
# your code here


### Exercise 5 — `evaluate_gate1(summary, thresholds)` (the boss fight)

This combines everything: a function that takes a results dict (from Exercise 4) and a thresholds dict (from Exercise 3), and returns a **new dict** indicating which thresholds passed.

Specifically, write a function `evaluate_gate1(summary, thresholds)` that returns a dict with these keys:
- `pf_pass` — True if `summary["profit_factor"] >= thresholds["min_pf"]`
- `total_pnl_pass` — True if `summary["total_pnl"] >= thresholds["min_total_pnl"]`
- `n_trades_pass` — True if `summary["n_trades"] >= thresholds["min_trades"]`
- `all_pass` — True only if all three above are True

Test it by:
1. Calling `summarise_trades(...)` on `[100, -50, 200, -150, 80, -30, 120]` to get a summary.
2. Passing the summary plus your `gate1_thresholds` dict from Exercise 3 to `evaluate_gate1`.
3. Printing the result.

Predict whether `all_pass` will be True or False *before* running. (Hint: count the trades in the test list — does it pass `min_trades=30`?)

*Why this is the boss fight:* this is structurally how `engine/metrics.py::evaluate_thresholds` works. Real Gate 1 has more checks, but the shape is identical: take a summary dict, take a thresholds dict, return a pass/fail dict. Once you can write this, the gate code in the engine will read like English.

In [ ]:
# your code here


---
## When you're done

Tell me:
1. Whether the `print` vs `return` distinction is fully clear, or still something you have to think about.
2. Whether dicts feel intuitive yet (think: would you reach for one if you needed to bundle related values?), or still feel slightly foreign.
3. How the boss-fight exercise went. That one is structurally identical to real engine code.

Next up after this is **notebook 04 — pandas Series basics**, the start of Phase 2. That's where things shift from pure-Python loops to the vectorised style most of `engine/` actually uses. Functions and dicts carry forward — they don't go away, they just gain a powerful new partner.